# Mesozoic Labs: T. rex form study, Stage 2 training (Track B)

**Simulation only.** This notebook runs the training queue of the T. rex form study's Stage 2 on a CPU runtime and keeps its files on Google Drive. Follow the steps sheet, `hardware/trex_form_study/stage2/tb/colab_steps.txt` in the repository ([the file at the launch tag](https://github.com/kuds/mesozoic-labs/blob/s2-launch-1/hardware/trex_form_study/stage2/tb/colab_steps.txt)); the lead sends its final text, which gives every value of the first cell and every line to expect.

**Nothing here trains in the notebook's own process.** Each code cell starts one step of `python -m designlab.stage2.nb` in a subprocess and prints its lines; the last cell releases the runtime. Run the cells in order (Runtime > Run all). If a cell stops with an error, nothing after it runs: read the lines above the error and the steps sheet, and change no parameter on an error's advice.

Runtime type: CPU with High-RAM, no GPU (Runtime > Change runtime type).

In [ ]:
# ===== 1. PARAMETERS: the steps sheet gives every value; change nothing else =====
REPO_REF = ""  # @param {"type":"string"}
RUN_ID = ""  # @param {"type":"string"}
QUEUE = []  # @param {"type":"raw"}
BUNDLE_ID = ""  # @param {"type":"string"}
BUNDLE_SOURCE = "drive"  # @param {"type":"string"}
RESUME = False  # @param {"type":"boolean"}
TAKEOVER = False  # @param {"type":"boolean"}
SMOKE = False  # @param {"type":"boolean"}
BUDGET_STEPS = 0  # @param {"type":"integer"}
WORKERS = 0  # @param {"type":"integer"}
CLIPS = True  # @param {"type":"boolean"}
ALLOW_VERSION_CHANGE = False  # @param {"type":"boolean"}
ALLOW_SOURCE_CHANGE = False  # @param {"type":"boolean"}
USE_GOOGLE_DRIVE = True  # @param {"type":"boolean"}
AUTO_DISCONNECT = True  # @param {"type":"boolean"}
# REPO_REF: the 40 hex digits of the launch commit.  QUEUE: a list such as ["R5:75-M-H-10:42", "R6:75-M-H-10:42"].
# WORKERS: 0 means one per vCPU, at most 8.  RESUME, TAKEOVER and the two ALLOW parameters stay False unless the
# steps sheet (RESUME) or the lead (the other three) says otherwise.

import json
import os

S2_NAMES = ("REPO_REF", "RUN_ID", "QUEUE", "BUNDLE_ID", "BUNDLE_SOURCE", "RESUME", "TAKEOVER", "SMOKE", "BUDGET_STEPS",
            "WORKERS", "CLIPS", "ALLOW_VERSION_CHANGE", "ALLOW_SOURCE_CHANGE", "USE_GOOGLE_DRIVE", "AUTO_DISCONNECT")
PARAMS = {name: globals()[name] for name in S2_NAMES}
if os.environ.get("S2_PARAMS"):  # local runs and tests: a JSON file whose values replace the ones above
    with open(os.environ["S2_PARAMS"], encoding="utf-8") as _f:
        PARAMS.update(json.load(_f))
    globals().update({name: PARAMS[name] for name in S2_NAMES})
print("Parameters of this session:")
for _name in S2_NAMES:
    print(f"  {_name} = {PARAMS[_name]!r}")

In [ ]:
# ===== 2. SETUP: the checkout of REPO_REF, the pinned packages, line 1 =====
import pathlib
import re
import subprocess
import sys
import time

SETUP_STARTED = time.time()
IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
# No MUJOCO_GL is set for this kernel: nothing here imports MuJoCo, and every step below is a subprocess that is
# given its whole environment.

if IN_COLAB:
    repo_dir = pathlib.Path("/content/mesozoic-labs")
    if not REPO_REF.strip() or REPO_REF.startswith("-"):
        raise ValueError("REPO_REF must name a branch, tag, or commit.")
    if not SMOKE and not re.fullmatch(r"[0-9a-f]{40}", REPO_REF):
        raise ValueError("A launch needs REPO_REF as the 40 hex digits of the launch commit (the steps sheet gives it). Nothing was changed.")
    fresh_checkout = not repo_dir.exists()
    if fresh_checkout:
        subprocess.run(["git", "clone", "--no-checkout", "https://github.com/kuds/mesozoic-labs.git", str(repo_dir)], check=True)
    if not (repo_dir / ".git").exists():
        raise RuntimeError(f"{repo_dir} is not a Git checkout; use a fresh runtime or a different directory.")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=repo_dir, check=True)
    requested_commit = subprocess.check_output(["git", "rev-parse", "FETCH_HEAD^{commit}"], cwd=repo_dir, text=True).strip()
    current_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip()
    if fresh_checkout or not (repo_dir / ".git" / "index").exists() or current_commit != requested_commit:
        if any(name.split(".")[0] in ("environments", "designlab") for name in sys.modules):
            raise RuntimeError("REPO_REF changed after repository modules were imported. Restart the runtime, then rerun setup.")
        if (repo_dir / ".git" / "index").exists() and subprocess.check_output(["git", "status", "--porcelain"], cwd=repo_dir, text=True).strip():
            raise RuntimeError("The checkout has local edits. Save them before changing REPO_REF, or use a fresh runtime; no edits were discarded.")
        subprocess.run(["git", "checkout", "--detach", requested_commit], cwd=repo_dir, check=True)
    print(f"Repository ref: {REPO_REF}; commit: {requested_commit}")
    STUDY = str(repo_dir / "hardware" / "trex_form_study")
    S2_PARAMS_FILE = "/content/s2_params.json"
    LOCAL = "/content/s2runs"
else:
    STUDY = PARAMS.get("STUDY_DIR") or os.getcwd()
    S2_PARAMS_FILE = (os.environ.get("S2_PARAMS") or os.path.join(STUDY, "stage2", "tb", "work", "s2_params.json")) + ".resolved.json"
    LOCAL = PARAMS.get("LOCAL_DIR") or os.path.join(STUDY, "stage2", "tb", "work", "s2runs")
    print(f"Running locally; using the checkout at {STUDY}. REPO_REF selects only the Colab checkout.")

# The environment of every step, passed in full: no OpenGL, one BLAS thread, a fixed hash seed, the study directory
# on the path.  A step is a subprocess, so the freshly pinned packages are the ones it imports, and a crash in it is
# an exit code, not a dead kernel.
S2_ENV = dict(os.environ, MUJOCO_GL="disable", PYTHONPATH=".", OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1",
              MKL_NUM_THREADS="1", PYTHONHASHSEED="0", PYTHONUNBUFFERED="1", PYTHONDONTWRITEBYTECODE="1",
              GIT_OPTIONAL_LOCKS="0")
for _name in ("PYOPENGL_PLATFORM", "MPLBACKEND"):
    S2_ENV.pop(_name, None)


def s2_step(step, check=True):
    """Run one step of designlab.stage2.nb in a subprocess and print its lines as they come; its exit code."""
    os.makedirs(os.path.dirname(S2_PARAMS_FILE), exist_ok=True)
    with open(S2_PARAMS_FILE, "w", encoding="utf-8") as f:
        json.dump(PARAMS, f, indent=1)
    proc = subprocess.Popen([sys.executable, "-m", "designlab.stage2.nb", step, "--params", S2_PARAMS_FILE], cwd=STUDY,
                            env=S2_ENV, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        proc.terminate()  # the step ends its job and leaves the run as it is: it can be resumed
        proc.wait()
        raise
    code = proc.wait()
    if check and code != 0:
        raise RuntimeError(f"S2 step {step!r} stopped with exit code {code}: nothing after it runs. Read the lines above this error; the steps sheet says what to do.")
    return code


PARAMS.update(IN_COLAB=IN_COLAB, SETUP_STARTED=SETUP_STARTED, SETUP_RECORD=S2_PARAMS_FILE + ".setup.json",
              LOCAL_DIR=LOCAL, STUDY_DIR=STUDY)
s2_step("setup")

In [ ]:
# ===== 3. STORAGE AND SESSION: Google Drive, the run directory, one writer per run =====
if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/mesozoic-labs/stage2"
    os.makedirs(DRIVE, exist_ok=True)
    print(f"Google Drive mounted. Stage 2 files are under: {DRIVE}")
elif IN_COLAB:
    DRIVE = "/content/s2drive"
    print(f"Warning: USE_GOOGLE_DRIVE is False. The run's files go to {DRIVE} and are lost with the runtime.")
else:
    DRIVE = PARAMS.get("DRIVE_DIR") or os.path.join(STUDY, "stage2", "tb", "work", "drive")
    print(f"Running locally: {DRIVE} stands in for Drive.")
PARAMS["DRIVE_DIR"] = DRIVE
s2_step("session")

In [ ]:
# ===== 4. INPUTS: the bundle's parts from Drive, every file against its sha256, line 2 =====
s2_step("inputs")

In [ ]:
# ===== 5. PREFLIGHT: an archive loads, a policy builds, every env builds, G8a per body, line 3 =====
s2_step("preflight")

In [ ]:
# ===== 6. SMOKE: only when SMOKE is True (the day-one checks; no training job) =====
if SMOKE:
    s2_step("smoke")
else:
    print("SMOKE is False: this session runs no smoke.")

In [ ]:
# ===== 7. LAUNCH: the queue, one job at a time; line 4 per job; files copied to Drive every minute =====
if SMOKE:
    print("SMOKE is True: this session queues no training job.")
else:
    s2_step("launch", check=False)  # a failure does not raise here: it is written to HALT.json, and the last cell still runs

In [ ]:
# ===== 8. FINISH: the summary, the last copy, then the runtime is released =====
try:
    s2_step("finish", check=False)
finally:
    if STUDY not in sys.path:
        sys.path.append(STUDY)
    from designlab.stage2 import nb as s2_nb  # standard library only

    s2_nb.release_runtime(f"Stage 2 session of run {RUN_ID!r} is over.", in_colab=IN_COLAB, auto=AUTO_DISCONNECT,
                          flush_drive=USE_GOOGLE_DRIVE and IN_COLAB)